In [31]:
import pandas as pd

In [32]:
df = pd.read_csv('data.csv', encoding='latin1')
df.head()

C:\Users\Asus\AppData\Local\Temp\ipykernel_20576\3839603254.py:1: DtypeWarning: Columns (0: stn_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data.csv', encoding='latin1')


,stn_code,sampling_date,state,location,agency,type,so2,no2,rspm,spm,location_monitoring_station,pm2_5,date
0,150.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",4.8,17.4,NaN,NaN,NaN,NaN,1990-02-01
1,151.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,Industrial Area,3.1,7.0,NaN,NaN,NaN,NaN,1990-02-01
2,152.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",6.2,28.5,NaN,NaN,NaN,NaN,1990-02-01
3,150.0,March - M031990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",6.3,14.7,NaN,NaN,NaN,NaN,1990-03-01
4,151.0,March - M031990,Andhra Pradesh,Hyderabad,NaN,Industrial Area,4.7,7.5,NaN,NaN,NaN,NaN,1990-03-01


In [33]:
df.columns

Index(['stn_code', 'sampling_date', 'state', 'location', 'agency', 'type',
       'so2', 'no2', 'rspm', 'spm', 'location_monitoring_station', 'pm2_5',
       'date'],
      dtype='str')

In [34]:
df.dtypes

stn_code                        object
sampling_date                      str
state                              str
location                           str
agency                             str
type                               str
so2                            float64
no2                            float64
rspm                           float64
spm                            float64
location_monitoring_station        str
pm2_5                          float64
date                               str
dtype: object

In [35]:
df.isna().sum()

stn_code                       144077
sampling_date                       3
state                               0
location                            3
agency                         149481
type                             5393
so2                             34646
no2                             16233
rspm                            40222
spm                            237387
location_monitoring_station     27491
pm2_5                          426428
date                                7
dtype: int64

In [36]:
df=df.drop(['stn_code','agency','location_monitoring_station','pm2_5'],axis=1)

In [37]:
#remove the NaN Records in location
df = df.dropna(subset=['location'])

In [38]:
#fill the emty data by mode(for object data) and mean(for int data)
df['type']=df['type'].fillna(df['type'].mode()[0])
df['so2']=df['so2'].fillna(df['so2'].mean())
df['no2']=df['no2'].fillna(df['no2'].mean())
df['rspm']=df['rspm'].fillna(df['rspm'].mean())
df['spm'] = df['spm'].fillna(df['spm'].mean())

In [39]:
df.isna().sum()

sampling_date    0
state            0
location         0
type             0
so2              0
no2              0
rspm             0
spm              0
date             4
dtype: int64

In [40]:
#calculate the AQI using s02,no2,rspm,spm column 

def calculate_aqi(row):
    def sub_index(value, breakpoints):
        for (bp_lo, bp_hi, aqi_lo, aqi_hi) in breakpoints:
            if bp_lo <= value <= bp_hi:
                return ((aqi_hi - aqi_lo) / (bp_hi - bp_lo)) * (value - bp_lo) + aqi_lo
        return None

    so2_bp  = [(0,40,0,50),(41,80,51,100),(81,380,101,200),(381,800,201,300),(801,1600,301,400),(1601,2620,401,500)]
    no2_bp  = [(0,40,0,50),(41,80,51,100),(81,180,101,200),(181,280,201,300),(281,400,301,400),(401,800,401,500)]
    rspm_bp = [(0,30,0,50),(31,60,51,100),(61,90,101,200),(91,120,201,300),(121,250,301,400),(251,350,401,500)]
    spm_bp  = [(0,50,0,50),(51,100,51,100),(101,250,101,200),(251,350,201,300),(351,450,301,400),(451,600,401,500)]

    indexes = []
    for val, bp in [(row['so2'], so2_bp), (row['no2'], no2_bp),
                    (row['rspm'], rspm_bp), (row['spm'], spm_bp)]:
        si = sub_index(val, bp)
        if si: indexes.append(si)

    return round(max(indexes)) if indexes else None

df['aqi'] = df.apply(calculate_aqi, axis=1)
print(df['aqi'].describe())

count    434979.000000
mean        240.417685
std          95.321063
min           2.000000
25%         181.000000
50%         225.000000
75%         314.000000
max         500.000000
Name: aqi, dtype: float64


In [41]:
#give this AQI value to the label like aui<50 good

def aqi_label(aqi):
    if aqi <= 50:     return 'Good'
    elif aqi <= 100:  return 'Satisfactory'
    elif aqi <= 200:  return 'Moderate'
    elif aqi <= 300:  return 'Poor'
    elif aqi <= 400:  return 'Very Poor'
    else:             return 'Severe'

df['aqi_status'] = df['aqi'].apply(aqi_label)
print(df['aqi_status'].value_counts())

aqi_status
Moderate        168267
Very Poor       111925
Poor            101114
Satisfactory     25602
Severe           24328
Good              4503
Name: count, dtype: int64


In [42]:
df.head()

,sampling_date,state,location,type,so2,no2,rspm,spm,date,aqi,aqi_status
0,February - M021990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",4.8,17.4,108.832784,220.78348,1990-02-01,262.0,Poor
1,February - M021990,Andhra Pradesh,Hyderabad,Industrial Area,3.1,7.0,108.832784,220.78348,1990-02-01,262.0,Poor
2,February - M021990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",6.2,28.5,108.832784,220.78348,1990-02-01,262.0,Poor
3,March - M031990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",6.3,14.7,108.832784,220.78348,1990-03-01,262.0,Poor
4,March - M031990,Andhra Pradesh,Hyderabad,Industrial Area,4.7,7.5,108.832784,220.78348,1990-03-01,262.0,Poor


In [43]:
#change the formate for date and add season name
df['date'] = pd.to_datetime(df['date'])

df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:       return 'Winter'
    elif month in [3, 4, 5]:      return 'Summer'
    elif month in [6, 7, 8, 9]:   return 'Monsoon'
    else:                         return 'Post-Monsoon'

df['season'] = df['month'].apply(get_season)

In [44]:
#remove the sample_date column
df.drop(columns=[ 'sampling_date'], inplace=True)

In [45]:
df.head()

,state,location,type,so2,no2,rspm,spm,date,aqi,aqi_status,year,month,quarter,season
0,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",4.8,17.4,108.832784,220.78348,1990-02-01,262.0,Poor,1990.0,2.0,1.0,Winter
1,Andhra Pradesh,Hyderabad,Industrial Area,3.1,7.0,108.832784,220.78348,1990-02-01,262.0,Poor,1990.0,2.0,1.0,Winter
2,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",6.2,28.5,108.832784,220.78348,1990-02-01,262.0,Poor,1990.0,2.0,1.0,Winter
3,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas",6.3,14.7,108.832784,220.78348,1990-03-01,262.0,Poor,1990.0,3.0,1.0,Summer
4,Andhra Pradesh,Hyderabad,Industrial Area,4.7,7.5,108.832784,220.78348,1990-03-01,262.0,Poor,1990.0,3.0,1.0,Summer


In [46]:
#convert string data to integer data
from sklearn.preprocessing import LabelEncoder
#for status_order
status_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
status_map = {s: i for i, s in enumerate(status_order)}
df['aqi_status'] = df['aqi_status'].map(status_map)
#for season
season_map = {'Winter': 0, 'Summer': 1, 'Monsoon': 2, 'Post-Monsoon': 3}
df['season'] = df['season'].map(season_map)

#for state, location, type
le = LabelEncoder()

df['state']    = le.fit_transform(df['state'])
df['location'] = le.fit_transform(df['location'])
df['type']     = le.fit_transform(df['type'])

In [47]:
#remove date coulmn because we have season
df.drop(columns=['date'], inplace=True)

In [53]:
#devide the data into train and test
from sklearn.model_selection import train_test_split
x=df.drop(['aqi','aqi_status'],axis=1)
#y= aqi_status because we want to do classification
y=df['aqi_status'] 
#using stratify=y devide the data same means if there is 0 in train data for 50 time and in test 0  will be 8 to 9.
#if we dont use stratify=y then maybe devide the data like in train 50 time 0 and in test only 1 time 0
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print("X_train:", x_train.shape)
print("X_test:", x_test.shape)


X_train: (348591, 11)
X_test: (87148, 11)


In [55]:
#prepare the evaluate function
from sklearn.metrics import classification_report,confusion_matrix,cohen_kappa_score,accuracy_score
def evaluate_model(y_test,y_pred):
    cm=confusion_matrix(y_test,y_pred)
    print('Confusion matrix:',cm)
    ac=accuracy_score(y_test,y_pred)
    print('Accuracy: ',ac)
    cr=classification_report(y_test,y_pred)
    print('Classification report',cr)
    print('Error rate: ',1-ac)
    kp=cohen_kappa_score(y_test,y_pred)
    print('Kappa score: ',kp)